# Семинар 11. Модель «трансформер», часть 2

Сравним два предложенных продолжения английской фразы с прогнозом DistilGPT2. Затем проверим, как RoPE и ALiBi учитывают расстояние между токенами и какие вычисления можно сохранить для следующего шага.

Для запуска достаточно CPU. При первом запуске токенизатор и веса DistilGPT2 загружаются из Hugging Face (около 350 МБ).

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

try:
    installed = version("transformers")
except PackageNotFoundError:
    installed = None
if installed != "4.57.6": 
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.57.6"])

import matplotlib.pyplot as plt
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from transformers import AutoModelForCausalLM, AutoTokenizer

# Векторный вывод графиков и PNG двойного разрешения для растровых просмотрщиков.
set_matplotlib_formats("svg", "retina")

use_cpu = False  # Поставьте True для работы на CPU.
device = "cpu"
if not use_cpu and torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = "mps"
elif not use_cpu and torch.cuda.is_available():
    device = "cuda"
print("Устройство:", device)

In [ ]:
checkpoint = "distilbert/distilgpt2"
revision = "2290a62682d06624634c1f46a6ad5be0f47f38aa"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint, revision=revision, use_safetensors=True, attn_implementation="eager"
).to(device).eval()

### Содержание

1. [Один шаг готовой GPT](#section-1)
2. [RoPE и относительный сдвиг](#section-2)
3. [ALiBi и расстояние до ключа](#section-3)
4. [KV-cache при генерации](#section-4)
5. [Стратегии декодирования](#section-5)

В заданиях заменяйте многоточия в комментариях `#` своим ответом. Под каждым вопросом указано, какой вывод или разбор использовать для проверки.

<a id="section-1"></a>

## 1. Один шаг готовой GPT

Вы набрали *I enjoy walking with my cute dog*. Сравним два продолжения этой фразы. Токен — элемент словаря, например слово, его часть или знак препинания. Введённые токены вместе образуют префикс. [Годой, гл. 11 · пример GPT-2](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter11.ipynb#scrollTo=qpfW8LWvPIp0)



Токенизатор разбивает фразу на токены и находит их номера в словаре. Модель вычисляет для каждой позиции по одной оценке на токен словаря. Эти оценки называют **логитами**. Логиты в каждой строке зависят только от токенов до этой позиции включительно. Softmax превращает оценки одной строки в вероятности с суммой 1.

**Ваш прогноз.** Какая строка логитов учитывает всю фразу? Запишите позицию и причину в первом комментарии ниже. Предложите два продолжения после *dog* и впишите их в кавычки в `my_continuations`. Перед английским словом поставьте пробел. Запустите ячейку и сравните вероятности первых токенов ваших продолжений.

In [ ]:
# Моя строка и почему: …
my_continuations = ["", ""]  # Мои два варианта после dog
prompt = "I enjoy walking with my cute dog"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.inference_mode():
    output = model(ids, use_cache=False)
print("Логиты [фраза, позиция, токен словаря]:", tuple(output.logits.shape))
last_logits = output.logits[0, -1].float().cpu()  # первая фраза, последняя позиция
probabilities = last_logits.softmax(-1)

for text in my_continuations:
    if text:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        token = tokens[0]
        print(f"Мой вариант {text!r} · первый токен {tokenizer.decode([token])!r}: "
              f"{probabilities[token]:.2%}")

probs, indices = probabilities.topk(5)
for token, prob in zip(indices.tolist(), probs.tolist()):
    print(f"{tokenizer.decode([token])!r:16} {prob:.2%}")
print("Самый вероятный:", tokenizer.decode(ids[0].tolist() + [indices[0].item()]))

**Как одна голова самовнимания вычисляет вектор для dog?** У каждой позиции есть вектор $h$. Чтобы обновить этот вектор, модель собирает сведения из той же последовательности. В блоке GPT за это отвечает слой самовнимания. В готовой модели первый такой модуль доступен как `model.transformer.h[0].attn` (класс `GPT2Attention`). Внутри слоя несколько голов вычисляют свои векторы параллельно. В одной голове три обученных преобразования делают из $h$ **запрос** $q$, **ключ** $k$ и **значение** $v$. Например, запрос позиции *dog* сравнивается с ключами уже доступных позиций. По результатам сравнения модель определяет, с каким весом взять значение каждой позиции. Их взвешенная сумма — выход одной головы для позиции *dog*.

Такие преобразования называют **проекциями**. Если записать $h$ строкой и для краткости опустить слагаемые смещения, то $q=hW_Q$, $k=hW_K$, $v=hW_V$. Параметры трёх матриц учатся вместе с моделью.



На верхней схеме укажите источник запроса, ключей и значений при переводе. Из каких токенов получены ключи и значения для *dog* в GPT?

Впишите источники ниже и раскройте «Сверить источники запроса, ключей и значений».

In [ ]:
# Перевод: запрос из … , ключи и значения из …
# GPT: для запроса dog доступны …

<details>
<summary>Сверить источники запроса, ключей и значений</summary>

Французское *dans le parc* подсказывает продолжение английского перевода. В слое перекрёстного внимания запрос приходит из декодера, а ключи и значения — из векторов французского текста. У готовой GPT все три величины получаются из векторов английского префикса. [Годой, гл. 9 · запросы и ключи декодера](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=SxB4b7PXFTKy)

</details>

**Почему строка walking предсказывает другой токен, чем строка dog?** Каузальная маска закрывает будущие позиции. Ноль на рисунке разрешает связь, а минус бесконечность закрывает её. Строка — запрос, столбец — ключ.



**Ваш прогноз.** Какие токены доступны запросу T2 (*walking*)? Какой токен следует за *walking* в исходной фразе? Впишите оба ответа в комментарий ниже. Выполните ячейку и сравните строку T2 с рисунком «Сверить строку T2 с рисунком».

In [ ]:
# Мой ответ: T2 видит … ; следующий токен текста — …

# Каждая строка логитов соответствует одному токену входа.
pieces = [tokenizer.decode([t]) for t in ids[0].tolist()]
for t, (current, following) in enumerate(zip(pieces, pieces[1:])):
    print(f"T{t}: вход {current!r:12} следующий токен текста {following!r}")
print(f"T{len(pieces)-1}: вход {pieces[-1]!r}; продолжаем по output.logits[0, -1]")

<details>
<summary>Сверить строку T2 с рисунком</summary>



При обучении метка в позиции $t$ — номер фактического следующего токена $x_{t+1}$. Для walking это with. Для продолжения после dog читаем последнюю строку логитов.

</details>

**Ещё один токен.** Допишите точку к префиксу и мысленно повторите полный проход. Сколько строк логитов получится?

Запишите число строк и номер нужной строки ниже, затем раскройте «Проверить форму».

In [ ]:
# Строк логитов: … ; для продолжения беру строку …

<details>
<summary>Проверить форму</summary>

Одна фраза, восемь позиций, 50 257 токенов словаря. Продолжение выбирается по восьмой строке.

</details>

<a id="section-2"></a>

## 2. RoPE и относительный сдвиг

В DistilGPT2 ко входу прибавляется вектор позиции. Попробуем другой способ. Хотим, чтобы при переносе запроса и ключа на семь позиций сохранялось их скалярное произведение внутри одной головы. В RoPE позиция задаёт угол поворота каждой пары координат запроса и ключа. У двух векторов фиксированной длины скалярное произведение определяется углом между ними.



**Ваш прогноз.** На трёх единичных окружностях изображены запрос и ключ. На каких двух рисунках угол между ними одинаков? В комментарии ниже укажите пары позиций и объясните, как повернулись оба вектора. Затем раскройте «Разобрать общий поворот» и сверьте первые две оценки в выводе `rotate_pairs` и `rope_score`.

In [ ]:
# Мой ответ: совпадут пары позиций … и …, потому что …

<details>
<summary>Разобрать общий поворот</summary>

**RoPE** (*Rotary Position Embedding*) кодирует позицию поворотом пар координат запроса и ключа. Что произойдёт, если приписать вектору номер позиции? Для соседних позиций 100 и 101 вклад этой координаты в скалярное произведение равен 10 100, а для 1000 и 1001 — уже 1 001 000. Расстояние одинаковое. Какое свойство поворота избавляет оценку от зависимости от общего сдвига?

**Почему поворот?** В исходном Transformer сдвиг позиции на один токен можно описать линейным преобразованием пары синуса и косинуса. Если записать пару как $(\cos t\omega,\sin t\omega)$, формулы сложения углов дают поворот на $\omega$. Авторы RoFormer поставили задачу уже для **скалярного произведения**: каждой позиции доступен свой номер, а оценка связи между двумя токенами должна зависеть от их взаимного расположения. [Vaswani et al., §3.5](https://arxiv.org/abs/1706.03762) · [Su et al., §3.1–3.2](https://arxiv.org/html/2104.09864)

Пусть один шаг вдоль текста действует на пару координат матрицей поворота $U$. Она сохраняет длины и углы и потому ортогональна: $U^\top=U^{-1}$. Обозначим число шагов через $i$. Тогда $i$ шагов дают $U^i$. Зафиксируем содержательные векторы $q,k$ и раскроем их скалярное произведение после преобразования:

$$
(U^i q)^\top (U^j k)
=q^\top (U^i)^\top U^j k
=q^\top U^{j-i}k.
$$

В последнем переходе сработало ровно одно свойство: $U^\top=U^{-1}$. Номера позиций сократились до разности. Возьмём в качестве $U$ поворот плоскости. В голове размерности $d$ каждая пара координат вращается в своей плоскости, на угол $t\omega_r$ в позиции $t$, где $\omega_r=10000^{-2r/d}$ и $r=0,\ldots,d/2-1$. В коде ниже это одно преобразование применяется к запросам и ключам. Значения $v_j$ передаются дальше без поворота.

[Годой, гл. 9 · синусоидальное позиционное кодирование](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=iAc0rhgBFTK5)

</details>

In [ ]:
def rotate_pairs(x, position):
    dim = x.shape[-1]
    assert dim % 2 == 0
    freq = 10000 ** (-torch.arange(0, dim, 2, dtype=x.dtype, device=x.device) / dim)
    angle = position * freq
    even, odd = x[..., ::2], x[..., 1::2]
    return torch.stack((even * angle.cos() - odd * angle.sin(),
                        even * angle.sin() + odd * angle.cos()), dim=-1).flatten(-2)

q = torch.tensor([1., 0., 2., 1.])
k = torch.tensor([0., 1., 1., 2.])

def rope_score(i, j):
    return rotate_pairs(q, i) @ rotate_pairs(k, j) / q.numel()**0.5

for i, j in [(5, 2), (12, 9), (12, 8)]:
    print(f"i={i:2}, j={j:2}, оценка до softmax={rope_score(i, j):.4f}")

torch.testing.assert_close(rope_score(5, 2), rope_score(12, 9))


<details>
<summary>Как четыре координаты образуют две плоскости</summary>



В коде координаты 0 и 1 образуют первую пару, 2 и 3 — вторую. Частота означает угол поворота за один шаг по тексту. [RoFormer, рисунок 1](https://arxiv.org/abs/2104.09864)

</details>

**Эксперимент с расстоянием.** Запрос и ключ — одинаковые единичные векторы. Оставим запрос на месте и повернём ключ на один радиан за каждый шаг назад. Где скалярное произведение больше — через три шага или через шесть? Под вопросом запишите прогноз для расстояний 0, 3 и 6. Выполните ячейку и сравните три числа в том же порядке.

In [ ]:
# Мой прогноз для расстояний 0, 3, 6: …
angles = torch.tensor([0., 3., 6.])
print("Скалярные произведения:", angles.cos().tolist())

<a id="section-3"></a>

## 3. ALiBi и расстояние до ключа

До штрафа за расстояние оценки всех доступных ключей равны нулю. Хотим дать более близкому больший вес. Для каждого запроса одна голова вычисляет вес каждого доступного ключа. Этим весом умножается соответствующий вектор значения. Сумма весов по ключам равна единице.



**Ваш прогноз.** Запрос находится в позиции 2. Вес каждого доступного ключа вдвое меньше веса его соседа справа, а сумма весов равна единице. В комментариях ниже запишите четыре веса в порядке ключей 0, 1, 2, 3 и номер закрытого ключа. Раскройте «Вывести штраф за шаг», затем проверьте строку «Веса по ключам 0, 1, 2, 3» в выводе.

In [ ]:
# Мои веса по ключам 0, 1, 2, 3: …
# Закрытый ключ: …

<details>
<summary>Вывести штраф за шаг</summary>

**ALiBi** (*Attention with Linear Biases*) прибавляет к масштабированному скалярному произведению запроса и ключа смещение, линейное по расстоянию. Теперь проверим различие при фиксированном содержании: фаза RoPE может вернуться близко к исходной, а вклад ALiBi в оценку убывает на одну и ту же величину с каждым шагом в прошлое. Авторы ALiBi хотели обучать модель на коротких последовательностях и применять к более длинным. Они предложили фиксированный для каждой головы штраф за расстояние. Пусть $i$ — позиция запроса, $j$ — позиция ключа, $d$ — число координат в одной голове. До штрафа за расстояние оценка равна $s_{ij}=q_i^\top k_j/\sqrt d$. За каждый шаг в прошлое вычтем одно и то же число $m>0$. Оно задаёт наклон прямой штрафа. Для доступного ключа $j\le i$ в softmax подаётся $s_{ij}-m(i-j)$. Штраф определён и для расстояний, которые не встречались при обучении. [Press et al., §3](https://arxiv.org/html/2108.12409)

Обозначим оценку с учётом смещения и маски через $u_{ij}$: для $j\le i$ это $s_{ij}-m(i-j)$, для $j>i$ это $-\infty$. В строке запроса $i$ вес ключа $j$ равен $\alpha_{ij}=e^{u_{ij}}/\sum_\ell e^{u_{i\ell}}$. В одной строке сумма весов по позициям ключей равна единице.

Посмотрим, откуда берётся числитель для доступного ключа. Ненормированный вес раскладывается в произведение двух множителей:

$$
\exp\!\big(s_{ij}-m(i-j)\big)
=\underbrace{e^{s_{ij}}}_{\text{сходство}}
\underbrace{(e^{-m})^{i-j}}_{\text{вклад расстояния}}.
$$

При равных оценках сходства каждый дополнительный шаг в прошлое умножает вес на одну и ту же величину $e^{-m}$. Для $m=\ln 2$ два соседних доступных ключа получат веса в отношении $2:1$ в пользу ближнего. Нормировка общая для всей строки, поэтому отношение сохраняется. У каждой головы свой наклон — величина штрафа за один шаг назад. Будущие позиции в обеих головах закрывает каузальная маска. 

[Годой, гл. 9 · от оценок сходства к весам ключей](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=QJhdbBGAFTKg)

</details>

In [ ]:
# Запрос в позиции 2, содержательные оценки всех ключей равны нулю.
steps_back = 2 - torch.arange(4)
slope = torch.log(torch.tensor(2.0))
scores_row = (-slope * steps_back).masked_fill(steps_back < 0, -torch.inf)
weights_row = scores_row.softmax(dim=0)
print("Веса по ключам 0, 1, 2, 3:", [round(x, 3) for x in weights_row.tolist()])
print("Вес ключа 1 / вес ключа 2:", float(weights_row[1] / weights_row[2]))
torch.testing.assert_close(weights_row, torch.tensor([1., 2., 4., 0.]) / 7)

**Две головы одного слоя самовнимания.** Штраф за шаг назад равен 0,15 в первой голове и 0,8 во второй. В какой голове запрос 7 даст больший вес ключу 7? Запишите прогноз в первом комментарии ниже. Выполните ячейку и сравните крайние правые числа в двух напечатанных строках. На верхних картах показаны оценки перед softmax, на нижних — веса после softmax.

In [ ]:
# Мой прогноз: ближайший ключ сильнее выделит голова с наклоном …

length = 8
pos = torch.arange(length)
distance = pos[:, None] - pos[None, :]  # [позиция запроса, позиция ключа]
slopes = torch.tensor([0.15, 0.8])
similarity_scores = torch.zeros(length, length)
biased_scores = similarity_scores - slopes[:, None, None] * distance
masked_scores = biased_scores.masked_fill(distance < 0, -torch.inf)
attention_weights = masked_scores.softmax(-1)

fig, axes = plt.subplots(2, 2, figsize=(8, 6), constrained_layout=True)
score_colors = plt.get_cmap("viridis", 48).with_extremes(bad="lightgray")
finite_scores = masked_scores[masked_scores.isfinite()]
positions = range(length)
for h, slope in enumerate(slopes):
    scores_plot = axes[0, h].pcolormesh(positions, positions, masked_scores[h], shading="nearest",
                                          cmap=score_colors, vmin=finite_scores.min(), vmax=finite_scores.max())
    weights_plot = axes[1, h].pcolormesh(positions, positions, attention_weights[h], shading="nearest",
                                            cmap=plt.get_cmap("Blues", 48), vmin=0, vmax=1)
    axes[0, h].set_title(f"Наклон {slope:.2f}")
    axes[1, h].set_xlabel("Позиция ключа")
for ax in axes.flat:
    ax.invert_yaxis()
    ax.set(aspect="equal", ylabel="Позиция запроса")
fig.colorbar(scores_plot, ax=axes[0], label="Оценка до softmax")
fig.colorbar(weights_plot, ax=axes[1], label="Вес ключа после softmax")
plt.show()
print("Веса в последней строке каждой головы:", attention_weights[:, -1].round(decimals=3))

**Может ли содержание перевесить расстояние?** Оставьте наклоны прежними и увеличьте оценку самого первого ключа на 2. В какой голове он теперь опередит ближайший ключ для запроса 7? Впишите наклон в комментарий ниже. Запустите ячейку и сравните номера ключей с наибольшим весом для двух голов.

In [ ]:
# Мой прогноз: ключ 0 станет лидером при наклоне …
bonus = 2.0
last_row = masked_scores[:, -1].clone()
last_row[:, 0] += bonus
print("Ключ с наибольшим весом в каждой голове:", last_row.argmax(dim=-1).tolist())

<a id="section-4"></a>

## 4. KV-cache при генерации

После четырёх токенов GPT получила пятый. Полный проход снова обрабатывает все пять позиций. Сравним полный расчёт с одним шагом генерации. В кэше каждого слоя самовнимания можно держать уже вычисленные ключи и значения. Эту память называют KV-cache.

**Ваш прогноз.** К четырём позициям дописали пятую. Что появится в новом столбце матрицы оценок у прежних запросов? Какие ключи и значения понадобятся пятому запросу? Запишите оба ответа ниже и раскройте «Почему прежние строки дают тот же результат».

In [ ]:
# Старые строки после добавления ключа: …
# Для пятого запроса потребуется: …

<details>
<summary>Почему прежние строки дают тот же результат</summary>

Рассмотрим одну голову в одном слое самовнимания GPT. Обозначим через $S_T$ матрицу оценок $QK^\top/\sqrt d$ после наложения каузальной маски на префикс длины $T$. Здесь $d$ — число координат у запроса и ключа одной головы. Добавим токен с запросом $q$, ключом $k$ и значением $v$. Если $q,k$ — столбцы, а старые ключи записаны строками матрицы $K_T$, получим

$$
S_{T+1}=
\begin{pmatrix}
S_T & -\infty\,\mathbf 1_T\\
q^\top K_T^\top/\sqrt d & q^\top k/\sqrt d
\end{pmatrix}.
$$

Softmax действует **по строкам**. Справа от каждой старой строки появился элемент с $e^{-\infty}=0$, поэтому и числители, и знаменатель для старых позиций прежние. Их взвешенные суммы значений сохраняются. В GPT нормализация и полносвязная сеть блока (MLP, у Годоя *feed-forward network*) работают отдельно в каждой позиции, так что этот аргумент повторяется слой за слоем в режиме оценки.

Отсюда и алгоритм. На первом проходе (*prefill*) сохраним $K_T,V_T$ каждого слоя. На шаге *decode* подадим один новый токен и дополним кэш. Для него вычислим нижнюю строку, применим softmax и возьмём взвешенную сумму старых $V_T$ и нового $v$. Старые запросы в этой формуле не участвуют. Если на входах одни и те же токены, обе схемы должны получить одинаковые логиты с точностью вычислений. [Shazeer, инкрементальное декодирование](https://arxiv.org/abs/1911.02150) · [Hugging Face, устройство KV-cache](https://huggingface.co/docs/transformers/v4.57.1/cache_explanation) Здесь шаг *decode* вычисляет логиты. Каким правилом выбрать из них следующий токен, решает стратегия декодирования. В проверке `argmax` берёт индекс максимального логита.

[Годой, гл. 9 · маска декодера при предсказании](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=EHchUewjFTK0)

</details>

<img alt="Префикс из четырёх позиций. Для пятой позиции полный проход вычисляет пять пар K и V, шаг с кэшем вычисляет одну пару и читает четыре сохранённые." width="660" src="">

Для первых четырёх позиций ключи и значения уже готовы. На пятом шаге новый запрос использует все пять ключей. **Prefill** — обработка исходного префикса и заполнение кэша. На последующем шаге **decode** модель обрабатывает один добавленный токен.

**Как проверить кэш вручную?** В задаче Годоя энкодеру даны вершины A и B, а декодер предсказывает C, затем D. В нашем опыте координаты вершины играют сразу три роли — запроса, ключа и значения. При обучении второй строке подают известную C. В проверке обе ветви получают эту же точку.



[Годой, гл. 9 · маска при обучении на вершинах квадрата](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=hHL1RKllFTKz)

**Ваш прогноз.** Какие ключи доступны запросу B? А запросу C? Сравните контекст для C при полном расчёте и при использовании сохранённых данных о B. Запишите доступные ключи и прогноз для контекста C в комментарии ниже. Запустите ячейку: `assert_close` сравнит два вектора, а вывод покажет веса ключей и контекст C.

In [ ]:
# Мой ответ: для B доступны … ; для C доступны … ; контексты для C …

corners = torch.tensor([[-1., -1.], [-1., 1.], [1., 1.], [1., -1.]])
decoder_points = corners[1:3]  # B и C; правильные следующие точки — C и D
scale = decoder_points.size(1) ** -0.5
scores = (decoder_points @ decoder_points.T) * scale
allowed = torch.ones_like(scores, dtype=torch.bool).tril()
weights = scores.masked_fill(~allowed, -torch.inf).softmax(-1)
full_context = weights @ decoder_points

past_k = decoder_points[:1]  # проекции тождественные, в кэше только B
past_v = decoder_points[:1]
new_point = decoder_points[1:]
new_q, new_k, new_v = new_point, new_point, new_point  # три тождественные проекции
keys = torch.cat((past_k, new_k))
values = torch.cat((past_v, new_v))
step_weights = ((new_q @ keys.T) * scale).softmax(-1)
torch.testing.assert_close(step_weights @ values, full_context[-1:])
print("Веса нового запроса по B и C:", step_weights.tolist())
print("Контекст нового запроса:", full_context[-1].tolist())

**Где на карте окажется подсказка из будущего?** Слева будет путь A → B → C → D, справа — две строки весов. В какой клетке первой строки декодер увидел бы правильный ответ заранее? Запишите строку и столбец в первом комментарии ниже. Выполните ячейку и найдите на правой карте клетку с подписью «закрыто».

In [ ]:
# Мой прогноз: закрытая клетка будет в строке … и столбце …
fig, (path_ax, weight_ax) = plt.subplots(1, 2, figsize=(8, 3.3), layout="constrained")
for i, (x, y) in enumerate(corners.tolist()):
    path_ax.text(x, y + 0.16, "ABCD"[i], ha="center", weight="bold")
for i, (start, end) in enumerate(zip(corners[:-1], corners[1:])):
    path_ax.annotate("", xy=end.tolist(), xytext=start.tolist(),
                     arrowprops=dict(arrowstyle="->", lw=2, color=["#126a66", "#b56815", "#b56815"][i]))
path_ax.set(xlim=(-1.4, 1.4), ylim=(-1.35, 1.5), aspect="equal",
            xticks=[-1, 1], yticks=[-1, 1], title="A, B известны → предсказать C, D")
weight_ax.pcolormesh([0, 1], [0, 1], weights, shading="nearest", cmap="Blues", vmin=0, vmax=1)
weight_ax.invert_yaxis()
weight_ax.set_aspect("equal")
for i in range(2):
    for j in range(2):
        label = f"{weights[i, j]:.2f}" if allowed[i, j] else "закрыто"
        weight_ax.text(j, i, label, ha="center", va="center",
                       color="white" if weights[i, j] > .6 else "#152d49")
weight_ax.set(xticks=[0, 1], xticklabels=["B", "C"], xlabel="Ключ",
              yticks=[0, 1], yticklabels=["B → C", "C → D"], ylabel="Вход → метка",
              title="Веса ключей B и C")
plt.show()

**Проверить на знакомой геометрии.** Как изменится отношение весов B и C при штрафе $m=\ln 2$? Что произойдёт с оценкой пары C и B при общем сдвиге позиций в RoPE?

Запишите оба изменения ниже и раскройте «Сверить рассуждение».

In [ ]:
# ALiBi: отношение веса B к весу C умножится на …
# RoPE: скалярное произведение после общего сдвига …

<details>
<summary>Сверить рассуждение</summary>

В строке C дальний ключ B получает множитель $e^{-m}=1/2$ по отношению к C. В RoPE общий сдвиг сохраняет разность позиций и скалярное произведение.

</details>

**Сохранился ли прогноз настоящей GPT?** На каждом шаге обе версии получают одно и то же продолжение. Одна пересчитывает весь текст, вторая берёт из памяти старые K и V. Как проверить совпадение прогнозов, если у двух разных векторов логитов может быть один и тот же лидер? В выводе `max |Δ|` — наибольшее расхождение среди всех токенов словаря. Запишите критерий в комментарии ниже. Запустите ячейку: для каждого из четырёх шагов сравните длину кэша и максимальное расхождение логитов.

In [ ]:
# Мой критерий совпадения двух проходов: …

@torch.inference_mode()
def compare_cache(prefix, steps=4):
    full_ids = prefix.clone()
    cached = model(full_ids, use_cache=True)
    cache = cached.past_key_values
    for step in range(1, steps + 1):
        token = cached.logits[:, -1].argmax(-1, keepdim=True)
        full_ids = torch.cat((full_ids, token), dim=1)
        reference = model(full_ids, use_cache=False).logits[:, -1]
        cached = model(token, past_key_values=cache, use_cache=True)
        cache = cached.past_key_values
        actual = cached.logits[:, -1]
        diff = (reference - actual).abs().max().item()
        print(f"Шаг {step}: вход {full_ids.size(1)} / 1 токен, "
              f"длина кэша {cache.get_seq_length()}, max |Δ| = {diff:.2e}")
        assert cache.get_seq_length() == full_ids.size(1)
        torch.testing.assert_close(reference, actual, rtol=1e-4, atol=3e-4)
    return cache

cache = compare_cache(ids)
keys, values = cache[0]
print("Первый слой, K и V:", keys.shape, values.shape)

**Сколько памяти добавляет один токен в кэше?** У GPT шесть слоёв. В каждом слое для каждой позиции хранятся два набора чисел, K и V. Форма каждого тензора — [число фраз, число голов, длина текста, число координат одной головы]. Для одной фразы, 12 голов, 64 координат и четырёх байт на число запишите произведение и оцените прирост в КиБ. Один КиБ равен 1024 байтам. Впишите расчёт в комментарий ниже, запустите ячейку и сравните свой ответ со строкой «Прирост на позицию».

In [ ]:
# Мой расчёт на одну позицию: … байт = … КиБ

bytes_per_token = 2 * model.config.n_layer * keys.size(0) * keys.size(1) * keys.size(3) * keys.element_size()
print(f"Прирост на позицию: {bytes_per_token:,} байт")
for length in (32, 256, 1024):
    print(f"{length:4} токенов: {length * bytes_per_token / 2**20:.2f} МиБ для K и V")

<a id="section-5"></a>

## 5. Стратегии декодирования

Оба способа вычисления дали одинаковые логиты в пределах численной погрешности. Осталось выбрать следующий токен. Жадный выбор (*greedy*) берёт лидера. При случайной выборке (*сэмплировании*) шанс токена равен его вероятности. Температура — положительное число, на которое делят логиты перед softmax.



**Ваш прогноз.** При температуре 1 токен A выпадает вдвое чаще B. Если уменьшить температуру до ½, каким станет отношение вероятностей A и B? Кто останется лидером? Запишите отношение и более вероятный токен в комментарии ниже. Раскройте «Вывести отношение вероятностей», затем сравните столбцы P(A) и A/B в выводе кода.

In [ ]:
# Мой ответ: отношение P(A)/P(B) = … ; лидер — …

<details>
<summary>Вывести отношение вероятностей</summary>

**Что именно регулирует температура?** Пусть $z_a,z_b$ — логиты двух токенов, а $P_\tau=\mathrm{softmax}(z/\tau)$ при $\tau>0$. В отношении вероятностей нормирующая сумма сокращается:

$$
\frac{P_\tau(a)}{P_\tau(b)}
=\exp\!\left(\frac{z_a-z_b}{\tau}\right).
$$

Если при $\tau=1$ один токен вдвое вероятнее другого, то при $\tau=1/2$ он станет вчетверо вероятнее. Порядок токенов сохраняется. При единственном максимуме и $\tau\to0^+$ вся масса стремится к токену, который выбрал бы greedy.

</details>

In [ ]:
# Эти логиты при температуре 1 дают исходное распределение из рисунка.
toy_logits = torch.tensor([2., 1.]).log()
for temperature in (1.0, 0.5):
    probabilities = (toy_logits / temperature).softmax(dim=0)
    odds = probabilities[0] / probabilities[1]
    print(f"τ = {temperature}: P(A) = {probabilities[0]:.3f}, "
          f"P(B) = {probabilities[1]:.3f}, A/B = {odds:.1f}")
torch.testing.assert_close(odds, torch.tensor(4.))

**Сколько продолжений оставить?** Top-k сохраняет заданное число самых вероятных токенов. Top-p сохраняет минимальное число лидеров, набирающих заданную массу. Масса — сумма их исходных вероятностей.



**Ваш прогноз.** Какую массу сохранят два лидера на каждой панели? Сколько токенов нужно слева и справа при пороге 0,80? Запишите ответы для обеих диаграмм в двух комментариях ниже. Сверьте правило в «Записать правило отсечения», затем запустите функцию `candidates` и следующую ячейку с двумя распределениями.

In [ ]:
# При top-k=2 массы слева и справа: …
# При top-p=0.8 число токенов слева и справа: …

<details>
<summary>Записать правило отсечения</summary>

**Top-k и top-p (nucleus sampling).** Вероятностная масса множества токенов — сумма их вероятностей. Зачем менять число кандидатов на каждом шаге? При открытой генерации Fan et al. выбирали следующий токен из $k$ самых вероятных. Holtzman et al. обратили внимание на изменение формы распределения от шага к шагу: иногда почти вся вероятность приходится на одно продолжение, иногда правдоподобных продолжений много. Они предложили сохранять заданную долю массы. Если $p_{(1)}\ge p_{(2)}\ge\cdots$ — вероятности в порядке убывания, размер множества кандидатов равен

$$
k_* = \min\left\{r:\sum_{j=1}^r p_{(j)}\ge p\right\}.
$$

После выбора множества вероятности внутри него делятся на его массу. На диаграмме ниже тот же выбор проделан для словаря DistilGPT2. [Fan et al., *Hierarchical Neural Story Generation*](https://arxiv.org/abs/1805.04833) · [Holtzman et al., *The Curious Case of Neural Text Degeneration*](https://arxiv.org/abs/1904.09751)

</details>

In [ ]:
def candidates(logits, temperature=1., top_k=None, top_p=None):
    if temperature <= 0 or (top_k is not None and not 1 <= top_k <= logits.numel()):
        raise ValueError("Проверьте температуру и число кандидатов")
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError("Порог top_p должен лежать в (0, 1]")
    prob, order = (logits / temperature).softmax(-1).sort(descending=True)
    if top_k is not None:
        prob, order = prob[:top_k], order[:top_k]
        prob = prob / prob.sum()
    if top_p is not None and top_p < 1:
        keep = prob.cumsum(0) - prob < top_p  # включает токен, пересекающий порог
        prob, order = prob[keep], order[keep]
    return order, prob / prob.sum()

In [ ]:
rows = torch.tensor([[.85, .05, .05, .05], [.25, .25, .25, .25]])
for row, size in zip(rows, [1, 4]):
    top_two, _ = candidates(row.log(), top_k=2)
    kept, normalized = candidates(row.log(), top_p=.8)
    print(f"Top-k: масса {row[top_two].sum():.2f} | Top-p: {len(kept)} ток., "
          f"масса {row[kept].sum():.2f}, после нормировки {normalized.sum():.2f}")
    assert len(kept) == size

**Те же правила на словаре GPT.** На диаграмме будут восемь первых токенов после *dog*. Если зафиксировать `top_k=5`, сколько из восьми полос исчезнет? Обязано ли число над панелью top-p совпасть с числом видимых полос? В комментарии ниже укажите число исчезнувших полос и объясните смысл счётчика. Запустите код и сравните панели Top-k и Top-p.

In [ ]:
# Мой прогноз: при top-k=5 из восьми исходных полос исчезнут …

strategies = [
    ("Исходное распределение", {}),
    ("Температура 0.6", {"temperature": 0.6}),
    ("Top-k = 5", {"top_k": 5}),
    ("Top-p = 0.8", {"top_p": 0.8}),
]
leaders = last_logits.topk(8).indices
labels = [repr(tokenizer.decode([t])) for t in leaders.tolist()]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.8), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, (title, params) in zip(axes, strategies):
    order, prob = candidates(last_logits, **params)
    distribution = torch.zeros_like(last_logits)
    distribution[order] = prob
    ax.barh(range(len(leaders)), distribution[leaders], color="#287789")
    ax.set_title(f"{title}\nКандидатов: {len(order):,}", fontsize=11)
    ax.set_xlabel("Вероятность")
axes[0].set_yticks(range(len(leaders)), labels)
axes[0].invert_yaxis()
plt.show()

<details>
<summary>Почему счётчик может быть больше числа полос</summary>

На каждой панели показаны те же восемь исходных лидеров. Счётчик относится ко всему словарю. После отсечения оставшиеся вероятности делятся на сумму сохранённых весов.

</details>

**Где разойдутся два продолжения одной фразы?** Сравните жадный выбор и случайную выборку из пяти лидеров. Начальная фраза и предел в 24 новых токена одинаковы. `seed` задаёт начальное состояние генератора случайных чисел. В начале следующей ячейки запишите, что получит модель после первого расхождения. Выполните код и сравните строки Greedy и Top-k до первого различающегося токена. [Годой, гл. 11 · Generating Text](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter11.ipynb#scrollTo=UYsl0aOiPIp5)

<details>
<summary>Самостоятельный опыт с генерацией</summary>

В строке с top-p задайте другой `seed` и найдите первое расхождение текстов. Для нового префикса измените `prompt` в первом опыте и повторите оттуда вычисления. Чтобы проверить влияние температуры на число кандидатов, измените параметры панели top-p в коде диаграммы, сохранив тот же вектор логитов.

</details>

In [ ]:
# Мой прогноз: после первого различия модель получит …

@torch.inference_mode()
def continuation(sample, seed=3, **params):
    torch.manual_seed(seed)
    generated = model.generate(
        ids, attention_mask=torch.ones_like(ids), max_new_tokens=24,
        pad_token_id=tokenizer.eos_token_id, do_sample=sample, **params
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True)

print("Greedy:", continuation(False))
print("Температура:", continuation(True, temperature=0.7, top_k=0))
print("Top-k:", continuation(True, top_k=5))
print("Top-p:", continuation(True, top_k=0, top_p=0.8))

**Распознавание текста на изображении (OCR).** Энкодер превращает изображение в ряд зрительных признаков. Декодер использует их как источник ключей и значений в слое перекрёстного внимания и выдаёт один символ за шаг. SOS отмечает начало строки, EOS — её конец. PAD дополняет короткую строку до длины остальных в пакете. При обучении входом следующего шага служит правильный предыдущий символ — этот режим называют *teacher forcing*. [Годой, гл. 9 · Teacher Forcing](https://colab.research.google.com/github/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb#scrollTo=KAYbSnqCFTKT)

**Ваш ответ для двух подписей.** Для «А7» и «7» выпишите входы декодера и символы, которые он должен предсказать. Какой символ обозначает конец строки? Какую позицию второй строки исключают из расчёта функции потерь? Заполните два комментария ниже. Выполните код и сверьте обе пары последовательностей и маску позиций.

In [ ]:
# «А7»: вход … → метки …
# «7»:  вход … → метки … ; пропустить позицию …

# Порядок служебных токенов взят из RecognitionDataset семинара 9.
SOS, EOS, PAD = 0, 1, 2
a, seven = 3, 4  # условные индексы двух символов учебного примера
labels = torch.tensor([[SOS, a, seven, EOS], [SOS, seven, EOS, PAD]])
decoder_input = labels[:, :-1]
targets = labels[:, 1:]
for row, (prefix, target) in enumerate(zip(decoder_input, targets), start=1):
    print(f"Строка {row}: вход {prefix.tolist()}, метки {target.tolist()}")
print("Позиции для расчёта функции потерь:", targets.ne(PAD).tolist())

**Перенос в прошлогоднее ДЗ № 5.** Изображение одно, а распознанный текст растёт. В [семинаре 9](https://github.com/SergeyMalashenko/2026_ML3/blob/e4ed03f/notebooks/09_crnn_attention/seminar09_crnn_attention.ipynb) декодер получал зрительные признаки энкодера. Как меняется кэш ключей и значений символов в каждом слое самовнимания декодера? Можно ли повторно использовать ключи и значения изображения? Что следует зафиксировать при сравнении трёх OCR-моделей?

Запишите ответы ниже, затем раскройте «Сверить ответ».

In [ ]:
# Кэш символов: … ; ключи и значения изображения: …
# Для сравнения трёх моделей зафиксируем …

<details>
<summary>Сверить ответ</summary>

Кэш символов растёт на одну позицию за шаг. Признаки одного изображения постоянны, в каждом слое перекрёстного внимания можно сохранить вычисленные из них ключи и значения. Сравниваем модели на одних изображениях и разбиении, по доле целиком верно сгенерированных строк.

</details>

**Дополнительно.** Лучевой поиск (*beam search*) хранит несколько префиксов и их накопленные логарифмы вероятностей. Почему списка лучших символов только последнего шага недостаточно?

Запишите объяснение ниже и раскройте «Сверить ответ про луч».

In [ ]:
# Для каждого префикса нужно хранить … , потому что …

<details>
<summary>Сверить ответ про луч</summary>

Продолжение зависит от всего префикса. Один и тот же последний символ может завершать разные последовательности с разными суммарными оценками.

</details>

[Материал семинара 2025 года · GPT и дополнительные способы генерации](https://github.com/SergeyMalashenko/2025_ML3/blob/main/seminars/11_gpt/seminar11_gpt.ipynb)